In [ ]:
# ============================================================
# RailGuard — YOLO26
# Environment Setup
# ============================================================

!pip install -q ultralytics==8.4.173

import os
import sys
import shutil
import time
import pandas as pd
import torch

from pathlib import Path
from ultralytics import YOLO
import ultralytics

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Ultralytics:", ultralytics.__version__)

In [ ]:
# ============================================================
# PATH CONFIGURATION
# ============================================================

SOURCE_ROOT = Path(
    "/kaggle/input/datasets/balajis11556/raildataset/"
    "rail defects detection.v18i.yolov8"
)

WORK_ROOT = Path("/kaggle/working")

BINARY_ROOT = WORK_ROOT / "railguard_yolo26_dataset"

RESULTS_ROOT = WORK_ROOT / "railguard_yolo26"

MODEL_OUTPUT = (
    WORK_ROOT /
    "YOLO26_railway_defect_best.pt"
)

print("Source dataset:", SOURCE_ROOT)
print("Binary dataset:", BINARY_ROOT)
print("Results:", RESULTS_ROOT)
print("Final model:", MODEL_OUTPUT)

In [ ]:
# ============================================================
# CHECK ORIGINAL DATASET STRUCTURE
# ============================================================

print("Source exists:", SOURCE_ROOT.exists())
print("Source:", SOURCE_ROOT)

if not SOURCE_ROOT.exists():
    raise FileNotFoundError(
        f"Dataset not found: {SOURCE_ROOT}"
    )

for split in ["train", "valid", "test"]:

    images = SOURCE_ROOT / split / "images"
    labels = SOURCE_ROOT / split / "labels"

    print("\n" + "=" * 50)
    print(split.upper())
    print("=" * 50)

    print("Images directory:", images.exists())
    print("Labels directory:", labels.exists())

    if images.exists():
        print(
            "Images:",
            len([
                p for p in images.iterdir()
                if p.is_file()
            ])
        )

    if labels.exists():
        print(
            "Labels:",
            len(list(labels.glob("*.txt")))
        )

In [ ]:
# ============================================================
# CREATE SINGLE-CLASS BASELINE DATASET
#
# Original classes:
# 0, 1, 2, ... 9
#
# Current RailGuard task:
# 0 = defect
#
# No synthetic data is created.
# Original bounding-box coordinates are preserved.
# ============================================================

if BINARY_ROOT.exists():
    shutil.rmtree(BINARY_ROOT)

for split in ["train", "valid", "test"]:

    (
        BINARY_ROOT /
        split /
        "images"
    ).mkdir(parents=True, exist_ok=True)

    (
        BINARY_ROOT /
        split /
        "labels"
    ).mkdir(parents=True, exist_ok=True)


for split in ["train", "valid", "test"]:

    source_images = (
        SOURCE_ROOT /
        split /
        "images"
    )

    source_labels = (
        SOURCE_ROOT /
        split /
        "labels"
    )

    dest_images = (
        BINARY_ROOT /
        split /
        "images"
    )

    dest_labels = (
        BINARY_ROOT /
        split /
        "labels"
    )

    image_files = [
        p for p in source_images.iterdir()
        if p.is_file()
    ]

    for image in image_files:
        shutil.copy2(
            image,
            dest_images / image.name
        )

    converted_labels = 0
    empty_labels = 0

    for label in source_labels.glob("*.txt"):

        lines = label.read_text().splitlines()

        valid_lines = []

        for line in lines:

            parts = line.split()

            if len(parts) >= 5:

                valid_lines.append(
                    "0 " + " ".join(parts[1:5])
                )

        if valid_lines:

            output_label = (
                dest_labels /
                label.name
            )

            output_label.write_text(
                "\n".join(valid_lines) + "\n"
            )

            converted_labels += 1

        else:
            empty_labels += 1

    print("\n" + "=" * 50)
    print(split.upper())
    print("=" * 50)
    print("Images:", len(image_files))
    print("Non-empty labels:", converted_labels)
    print("Empty labels skipped:", empty_labels)

print("\n============================================")
print("Binary dataset created successfully.")
print("============================================")
print("Dataset:", BINARY_ROOT)

In [ ]:
# ============================================================
# REMOVE ORPHAN LABEL FILES
# ============================================================

for split in ["train", "valid", "test"]:

    images_dir = (
        BINARY_ROOT /
        split /
        "images"
    )

    labels_dir = (
        BINARY_ROOT /
        split /
        "labels"
    )

    image_stems = {
        image.stem
        for image in images_dir.iterdir()
        if image.is_file()
    }

    orphan_labels = []

    for label in labels_dir.glob("*.txt"):

        if label.stem not in image_stems:
            orphan_labels.append(label)

    for label in orphan_labels:
        label.unlink()

    print(split.upper())
    print("Orphan labels removed:", len(orphan_labels))
    print(
        "Remaining labels:",
        len(list(labels_dir.glob("*.txt")))
    )

In [ ]:
# ============================================================
# FINAL DATASET VERIFICATION
# ============================================================

dataset_summary = []

for split in ["train", "valid", "test"]:

    images_dir = (
        BINARY_ROOT /
        split /
        "images"
    )

    labels_dir = (
        BINARY_ROOT /
        split /
        "labels"
    )

    images = [
        p for p in images_dir.iterdir()
        if p.is_file()
    ]

    labels = list(
        labels_dir.glob("*.txt")
    )

    image_stems = {
        p.stem for p in images
    }

    label_stems = {
        p.stem for p in labels
    }

    missing_labels = (
        image_stems -
        label_stems
    )

    orphan_labels = (
        label_stems -
        image_stems
    )

    objects = 0
    empty_labels = 0

    for label in labels:

        lines = [
            line.strip()
            for line in label.read_text().splitlines()
            if line.strip()
        ]

        if not lines:
            empty_labels += 1

        objects += len(lines)

    dataset_summary.append({
        "split": split,
        "images": len(images),
        "labels": len(labels),
        "background_images": len(missing_labels),
        "orphan_labels": len(orphan_labels),
        "objects": objects,
        "empty_labels": empty_labels
    })

summary_df = pd.DataFrame(dataset_summary)

print(summary_df.to_string(index=False))

print("\nDataset integrity check:")
print(
    "Orphan labels:",
    summary_df["orphan_labels"].sum()
)

print(
    "Total objects:",
    summary_df["objects"].sum()
)

In [ ]:
# ============================================================
# CREATE YOLO DATASET YAML
# ============================================================

DATA_YAML = (
    BINARY_ROOT /
    "data.yaml"
)

yaml_content = f"""path: {BINARY_ROOT}
train: train/images
val: valid/images
test: test/images

nc: 1
names:
  0: defect
"""

DATA_YAML.write_text(
    yaml_content
)

print("data.yaml created:")
print("-" * 50)
print(DATA_YAML.read_text())

In [ ]:
# ============================================================
# VERIFY YOLO DATASET CONFIGURATION
# ============================================================

print("YAML exists:", DATA_YAML.exists())

print("\nDataset directories:")

for split in ["train", "valid", "test"]:

    image_dir = (
        BINARY_ROOT /
        split /
        "images"
    )

    label_dir = (
        BINARY_ROOT /
        split /
        "labels"
    )

    print(
        split.upper(),
        "| images:",
        image_dir.exists(),
        "| labels:",
        label_dir.exists()
    )

print("\nClass configuration:")
print("0: defect")

In [ ]:
# ============================================================
# LOAD YOLO26N
# ============================================================

model = YOLO("yolo26n.pt")

print("YOLO26n loaded successfully.")
print("Task: Object Detection")
print("Classes: 1")
print("Class 0: defect")

In [ ]:
# ============================================================
# YOLO26 TRAINING
# ============================================================

start_time = time.time()

train_results = model.train(

    data=str(DATA_YAML),

    epochs=50,
    imgsz=640,
    batch=16,

    optimizer="AdamW",

    device=0,

    amp=True,

    seed=0,

    deterministic=True,

    pretrained=True,

    project=str(RESULTS_ROOT),
    name="training",

    plots=True,
    save=True,

    workers=2
)

training_time = (
    time.time() -
    start_time
) / 3600

print("\n============================================")
print("YOLO26 TRAINING COMPLETE")
print("============================================")
print(
    f"Training time: "
    f"{training_time:.2f} hours"
)

print(
    "Training directory:",
    RESULTS_ROOT / "training"
)

In [ ]:
# ============================================================
# LOCATE BEST MODEL
# ============================================================

BEST_MODEL = (
    RESULTS_ROOT /
    "training" /
    "weights" /
    "best.pt"
)

LAST_MODEL = (
    RESULTS_ROOT /
    "training" /
    "weights" /
    "last.pt"
)

print("Best model:", BEST_MODEL)
print("Exists:", BEST_MODEL.exists())

print("\nLast model:", LAST_MODEL)
print("Exists:", LAST_MODEL.exists())

if BEST_MODEL.exists():

    size_mb = (
        BEST_MODEL.stat().st_size /
        (1024 * 1024)
    )

    print(
        f"\nBest model size: "
        f"{size_mb:.2f} MB"
    )

In [ ]:
# ============================================================
# SAVE FINAL YOLO26 MODEL
# ============================================================

if not BEST_MODEL.exists():
    raise FileNotFoundError(
        "best.pt was not found."
    )

shutil.copy2(
    BEST_MODEL,
    MODEL_OUTPUT
)

print("Final model saved:")
print(MODEL_OUTPUT)

print(
    f"Size: "
    f"{MODEL_OUTPUT.stat().st_size / (1024 * 1024):.2f} MB"
)

In [ ]:
# ============================================================
# VALIDATION EVALUATION
# ============================================================

best_model = YOLO(
    str(MODEL_OUTPUT)
)

validation_results = best_model.val(
    data=str(DATA_YAML),

    split="val",

    imgsz=640,
    batch=16,

    device=0,

    plots=True,

    project=str(RESULTS_ROOT),
    name="validation",
    exist_ok=True
)

print("Validation completed.")

In [ ]:
# ============================================================
# VALIDATION METRICS
# ============================================================

metrics = (
    validation_results.results_dict
)

precision = float(
    metrics.get(
        "metrics/precision(B)",
        0
    )
)

recall = float(
    metrics.get(
        "metrics/recall(B)",
        0
    )
)

map50 = float(
    metrics.get(
        "metrics/mAP50(B)",
        0
    )
)

map5095 = float(
    metrics.get(
        "metrics/mAP50-95(B)",
        0
    )
)

f1 = (
    2 * precision * recall /
    (precision + recall)
    if precision + recall > 0
    else 0
)

print("=" * 50)
print("YOLO26 VALIDATION RESULTS")
print("=" * 50)

print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")
print(f"mAP@50    : {map50:.4f}")
print(f"mAP@50-95 : {map5095:.4f}")

In [ ]:
# ============================================================
# TRAINING ARTIFACT VERIFICATION
# ============================================================

TRAINING_DIR = (
    RESULTS_ROOT /
    "training"
)

print("Training artifacts:")
print("=" * 50)

for item in sorted(
    TRAINING_DIR.iterdir()
):

    if item.is_file():
        print(item.name)

print("\nRequired artifacts:")

required = [
    "results.csv",
    "results.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "BoxP_curve.png",
    "BoxR_curve.png",
    "BoxF1_curve.png",
    "BoxPR_curve.png",
    "args.yaml"
]

for filename in required:

    path = TRAINING_DIR / filename

    print(
        filename,
        "->",
        "FOUND" if path.exists()
        else "MISSING"
    )

In [ ]:
# ============================================================
# FINAL TRAINING SUMMARY
# ============================================================

print("=" * 65)
print("RAILGUARD — YOLO26 TRAINING REPORT")
print("=" * 65)

print("\nMODEL")
print("-" * 40)
print("Architecture : YOLO26n")
print("Task         : Binary Object Detection")
print("Class        : defect")

print("\nDATASET")
print("-" * 40)
print("Source       : Mendeley Railway Defects Detection")
print("Train images : 6781")
print("Val images   : 606")
print("Test images  : 300")

print("\nTRAINING")
print("-" * 40)
print("Epochs       : 50")
print("Image size   : 640")
print("Batch size   : 16")
print("Optimizer    : AdamW")
print("Seed         : 0")
print(
    f"Training time: "
    f"{training_time:.2f} hours"
)

print("\nVALIDATION")
print("-" * 40)
print(f"Precision    : {precision:.4f}")
print(f"Recall       : {recall:.4f}")
print(f"F1-score     : {f1:.4f}")
print(f"mAP@50       : {map50:.4f}")
print(f"mAP@50-95    : {map5095:.4f}")

print("\nMODEL")
print("-" * 40)
print(MODEL_OUTPUT)

print("\nYOLO26 training workflow completed.")